# Figure: Shorkie ISM saliency + discovered motif logos (RP / TSS genes)

This figure has two parts. **(1)** A per-nucleotide ISM saliency DNA logo for one representative locus, where letter heights encode the in-silico-mutagenesis attribution (logSED, averaged over the T0 RNA-seq tracks) projected onto the reference base. Tall stacks mark positions whose mutation most changes Shorkie's predicted gene expression. **(2)** Sequence logos of the cis-regulatory motifs that TF-MoDISco discovers from those ISM contribution scores, summarizing the recurring grammar Shorkie uses around ribosomal-protein (RP) and transcription-start-site (TSS) genes.

**Reproduces:** the ISM DNA-logo panels and the TF-MoDISco motif-logo panels of the RP/TSS ISM-motif figure.

**Upstream:** the ISM `scores.h5` is produced by the ISM run under `scripts/04_analysis/shorkie/ism_motif/motif_shorkie__RP_TSS/ism_run/` (referenced here as `results.ism_scores`); the TF-MoDISco `modisco_results_*.h5` is produced by `scripts/04_analysis/shorkie/ism_motif/motif_shorkie__RP_TSS/2_modisco_analysis/1_process_modisco_input.py` followed by `2_modisco_script.sh` (referenced here as `results.modisco_ism`). Both are unreleased intermediates, so this notebook is **not** runnable end-to-end from released data.

**Requires:** the `yeast_ml` conda env with the package installed (`pip install -e .` from the repo root), plus `logomaker` for the motif logos. No GPU needed (only reads precomputed `.h5` artifacts). The reference FASTA (`genome.fasta`) is opened to recover the reference base for the saliency logo.

**Source script:** ported from `scripts/04_analysis/shorkie/ism_motif/motif_shorkie__RP_TSS/1_plot_dna_logo_general.py` (ISM saliency logo) and `scripts/04_analysis/shorkie/ism_motif/motif_shorkie__RP_TSS/2_modisco_analysis/4_viz_motif.py` (TF-MoDISco motif logos).

In [ ]:
import os
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import pysam
import logomaker

from shorkie import config
from shorkie.helpers.yeast_helpers import make_seq_1hot, plot_seq_scores

In [ ]:
# Resolve all paths through shorkie.config (no hardcoded absolute paths).
# ISM attributions (scores.h5) and the TF-MoDISco results live under the
# unreleased results.* keys; the orchestrator wires these into config.
ism_scores_dir = config.path('results.ism_scores')      # dir containing scores.h5
modisco_dir    = config.path('results.modisco_ism')     # dir containing modisco_results_*.h5
fasta_path     = str(config.path('genome.fasta'))       # R64 reference FASTA
targets_sheet  = str(config.path('datasets.targets_sheet'))

scores_h5  = os.path.join(str(ism_scores_dir), 'scores.h5')
modisco_h5 = os.path.join(str(modisco_dir), 'modisco_results_10000_500.h5')
print('ISM scores:   ', scores_h5)
print('MoDISco file: ', modisco_h5)

In [ ]:
# --- Part 1: select the T0 RNA-seq tracks the ISM was averaged over ---
# The score datasets are indexed by *local* track position; the released ISM run
# stored logSED for tracks starting at a global offset, so we subtract it to map
# the targets-sheet 'index' column into the score dataset's track axis.
track_offset = 1148

targets = pd.read_csv(targets_sheet, sep='\t')
t0 = targets[targets['identifier'].str.contains('_T0_')]
selected_tracks = [int(i) - track_offset for i in t0['index'].astype(int).tolist()]
print(f'Selected {len(selected_tracks)} T0 RNA-seq tracks for the ISM average.')

In [ ]:
# --- Part 1: build the per-base saliency PWM for one representative locus ---
# scores.h5['logSED'] has shape (N_sequences, L, 4, N_tracks); we average over the
# selected T0 tracks, mean-center across nucleotides, then project onto the
# reference one-hot so each position shows only the reference base's attribution.
fasta_open = pysam.Fastafile(fasta_path)
seq_idx = 0  # representative locus (first sequence in the ISM file)

with h5py.File(scores_h5, 'r') as h5:
    dataset = h5['logSED']
    L = dataset.shape[1]
    chrom = str(h5['chr'][seq_idx].decode('utf-8'))
    start = int(h5['start'][seq_idx])
    end   = int(h5['end'][seq_idx])
    pwm = dataset[seq_idx, :, :, np.array(selected_tracks, dtype=int)].mean(axis=-1)  # (L, 4)

print(f'Locus {chrom}:{start}-{end}  (length {L})')

# Mean-center across the 4 nucleotides at each position.
pwm_norm = pwm - pwm.mean(axis=-1, keepdims=True)

# Reference one-hot for this window, then project the saliency onto it.
seq_1hot = make_seq_1hot(fasta_open, chrom, start, end, end - start).astype('float32')
viz_pwm = pwm_norm * seq_1hot  # (L, 4) — nonzero only on the reference base
fasta_open.close()

In [ ]:
# --- Part 1: plot the ISM saliency DNA logo (zoomed to a readable window) ---
# plot_seq_scores draws each reference letter scaled by its summed attribution.
plot_start, plot_end = 0, min(120, viz_pwm.shape[0])
scores_win = viz_pwm[plot_start:plot_end, :]

y_abs = max(abs(scores_win.min()), abs(scores_win.max()))
plot_seq_scores(
    scores_win,
    figsize=(24, 3),
    plot_y_ticks=False,
    y_min=scores_win.min() - 0.05 * y_abs,
    y_max=scores_win.max() + 0.05 * y_abs,
)
plt.suptitle(f'ISM saliency (T0 logSED) — {chrom}:{start + plot_start}-{start + plot_end}', y=1.02)
plt.show()

In [ ]:
# --- Part 2: helpers to render TF-MoDISco contribution-weight-matrix logos ---
def compute_per_position_ic(ppm, background, pseudocount):
    alphabet_len = len(background)
    ic = ((np.log((ppm + pseudocount) / (1 + pseudocount * alphabet_len)) / np.log(2)) * ppm
          - (np.log(background) * background / np.log(2))[None, :])
    return np.sum(ic, axis=1)


def plot_cwm_logo(array, ax, title):
    """Plot a (positions, 4) CWM/PPM array as an A/C/G/T sequence logo."""
    df = pd.DataFrame(array, columns=['A', 'C', 'G', 'T'])
    df.index.name = 'pos'
    logo = logomaker.Logo(df, ax=ax)
    logo.style_spines(visible=False)
    ax.set_ylim(min(df.sum(axis=1).min(), 0), df.sum(axis=1).max())
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=10)

In [ ]:
# --- Part 2: load the top discovered motifs and plot their forward CWM logos ---
# Walk pos_patterns then neg_patterns, take the first few patterns, and draw the
# contribution-weight-matrix (contrib_scores) as a logo, exactly as 4_viz_motif.py.
pattern_groups = ['pos_patterns', 'neg_patterns']
max_patterns = 6

motifs = []  # list of (tag, cwm_fwd)
with h5py.File(modisco_h5, 'r') as modisco:
    for group in pattern_groups:
        if group not in modisco.keys():
            continue
        metacluster = modisco[group]
        for pattern_name, pattern in sorted(metacluster.items(), key=lambda x: int(x[0].split('_')[-1])):
            cwm_fwd = np.array(pattern['contrib_scores'][:])  # (positions, 4)
            motifs.append((f'{group}.{pattern_name}', cwm_fwd))
            if len(motifs) >= max_patterns:
                break
        if len(motifs) >= max_patterns:
            break

n = len(motifs)
fig, axes = plt.subplots(n, 1, figsize=(8, 2.2 * n))
if n == 1:
    axes = [axes]
for ax, (tag, cwm) in zip(axes, motifs):
    plot_cwm_logo(cwm, ax, tag)
fig.suptitle('TF-MoDISco motifs from Shorkie ISM (RP/TSS)', y=1.0)
plt.tight_layout()
plt.show()